# What the statements say

Companion notebook to *Advanced Calculus for Machine Learning*, across
chapters rather than for one of them.

The per-chapter notebooks recompute the numbers printed in the chapters, and
`conventions-and-identities.ipynb` checks the conventions the proofs rest on.
This one does something else: it takes the **statement** of a theorem,
re-implements it from its text alone, and looks for a counterexample on
random instances.

**What this can and cannot do.** A proof cannot be checked by a computer
here, and a thousand passing instances are not a proof. But a wrong
constant, a reversed inequality, a sign error, a missing hypothesis or a
rate that is too strong will all produce a counterexample, usually in the
first few draws. That is what this notebook is for: not confirming the
proofs, but making a mis-stated theorem fail loudly.

Each test reports the tightest margin it found over all its trials — the
smallest slack in the inequality, or the largest error in an identity.
A margin at the level of floating-point noise means the statement is
*exactly* attained somewhere in the sample, which is the interesting case:
the constant cannot be improved.

In [1]:
import numpy as np

rng = np.random.default_rng(20261120)

summary = []


def run(name, fn, trials=200):
    worst = None
    for t in range(trials):
        ok, margin = fn(rng)
        assert ok, f"{name}: counterexample at trial {t}, margin {margin:.3e}"
        if worst is None or margin < worst:
            worst = margin
    summary.append((name, trials, worst))
    print(f"ok   {name:46s} {trials:4d} trials, tightest margin {worst:.3e}")


def spd(rng, n, lo=0.3, hi=3.0):
    '''A symmetric positive definite matrix with eigenvalues in [lo, hi].'''
    Q = np.linalg.qr(rng.normal(size=(n, n)))[0]
    return Q @ np.diag(rng.uniform(lo, hi, n)) @ Q.T


def invertible(rng, n):
    '''A well-conditioned invertible matrix.'''
    A = rng.normal(size=(n, n))
    while np.linalg.cond(A) > 20:
        A = rng.normal(size=(n, n))
    return A


def grad_num(f, x, t=1e-5):
    '''A central-difference gradient.'''
    out = np.zeros(len(x))
    for i in range(len(x)):
        e = np.zeros(len(x))
        e[i] = t
        out[i] = (f(x + e) - f(x - e)) / (2 * t)
    return out


def rk4(F, y0, t0, t1, steps=4000):
    '''Classical Runge-Kutta, for the flows of Chapter 25.'''
    y, t = np.array(y0, dtype=float), float(t0)
    h = (t1 - t0) / steps
    for _ in range(steps):
        k1 = F(t, y)
        k2 = F(t + h / 2, y + h * k1 / 2)
        k3 = F(t + h / 2, y + h * k2 / 2)
        k4 = F(t + h, y + h * k3)
        y = y + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        t += h
    return y


def gauss(nodes=600, lo=0.0, hi=1.0):
    '''Gauss-Legendre nodes and weights on an interval.'''
    x, w = np.polynomial.legendre.leggauss(nodes)
    return (x + 1) / 2 * (hi - lo) + lo, w / 2 * (hi - lo)


def quad_problem(rng, n):
    '''A strongly convex quadratic, with mu and L known exactly.'''
    Q = np.linalg.qr(rng.normal(size=(n, n)))[0]
    lam = np.sort(rng.uniform(0.3, 3.0, n))
    H = Q @ np.diag(lam) @ Q.T
    c = rng.normal(size=n)
    f = lambda x: 0.5 * x @ H @ x + c @ x
    g = lambda x: H @ x + c
    return f, g, np.linalg.solve(H, -c), float(lam[0]), float(lam[-1])

## Chapters 3 and 4: inner products, norms, Hilbert spaces

Theorem 3.16 is the formula $M^{-1}A^\top N$ for the matrix of the adjoint
when the two metrics are not the identity — the formula the book insists on,
and the one that is wrong in most people's memory. Theorem 3.25 says that
the least-squares solutions closest to a given point are found by moving
inside $\operatorname{im} A^\*$, which is tested here against random
displacements in $\ker A$.

In [2]:
def t_adjoint_matrix(rng):
    """Thm 3.16: the matrix of the adjoint is M^-1 A^T N."""
    n, m = rng.integers(2, 6), rng.integers(2, 6)
    M = spd(rng, n)
    N = spd(rng, m)
    A = rng.normal(size=(m, n))
    Astar = np.linalg.solve(M, A.T @ N)
    v = rng.normal(size=n)
    w = rng.normal(size=m)
    lhs = (A @ v) @ N @ w
    rhs = v @ M @ (Astar @ w)
    return abs(lhs - rhs) <= 1e-9 * (1 + abs(lhs)), -abs(lhs - rhs)


def t_min_norm(rng):
    """Thm 3.25: S n (x0 + im A*) is one point, the point of S closest to x0.

    Euclidean V, W with metrics M, N; A* = M^-1 A^T N.
    S = argmin ||Ax - b||_N.
    """
    n, m = rng.integers(2, 6), rng.integers(2, 6)
    M = spd(rng, n)
    N = spd(rng, m)
    r = int(min(n, m) - 1) or 1
    A = rng.normal(size=(m, r)) @ rng.normal(size=(r, n))      # rank <= r
    b = rng.normal(size=m)
    x0 = rng.normal(size=n)
    Astar = np.linalg.solve(M, A.T @ N)
    # normal equations A* (A x - b) = 0
    G = Astar @ A
    rhs = Astar @ b
    # general solution: particular + ker A
    xp = np.linalg.lstsq(G, rhs, rcond=None)[0]
    # the claimed point: x0 + A*w solving the normal equations
    # A*(A(x0 + A* w) - b) = 0  ->  (A* A A*) w = A*(b - A x0)
    Kmat = Astar @ A @ Astar
    w = np.linalg.lstsq(Kmat, Astar @ (b - A @ x0), rcond=None)[0]
    xc = x0 + Astar @ w
    # (a) xc is a least-squares solution
    res = Astar @ (A @ xc - b)
    if np.linalg.norm(res) > 1e-6 * (1 + np.linalg.norm(rhs)):
        return False, -np.linalg.norm(res)
    # (b) no point of S is closer to x0 in the M-norm
    def dM(x):
        d = x - x0
        return float(d @ M @ d)
    base = dM(xc)
    ns = np.linalg.svd(A)[2][np.linalg.matrix_rank(A):]        # basis of ker A
    best = base
    for _ in range(40):
        z = ns.T @ rng.normal(size=ns.shape[0]) if ns.size else np.zeros(n)
        for s in (-1.0, -0.3, 0.3, 1.0):
            best = min(best, dM(xc + s * z))
    return best >= base - 1e-8 * (1 + base), base - best


run("Thm 3.16  matrix of the adjoint", t_adjoint_matrix)
run("Thm 3.25  the closest solution", t_min_norm)

ok   Thm 3.16  matrix of the adjoint                 200 trials, tightest margin -1.421e-14


ok   Thm 3.25  the closest solution                  200 trials, tightest margin 0.000e+00


In [3]:
def t_equivalence(rng):
    """Thm 4.17: on a finite-dimensional space any two norms are equivalent.

    Tested with the sharp constants: c = min and C = max of N2/N1 on the
    unit sphere of N1, found by sampling, must be finite and positive.
    """
    n = int(rng.integers(2, 6))
    M = spd(rng, n)
    n1 = lambda x: np.max(np.abs(x))                     # sup norm
    n2 = lambda x: np.sqrt(x @ M @ x)                    # a Euclidean norm
    lo, hi = np.inf, 0.0
    for _ in range(400):
        x = rng.normal(size=n)
        x = x / n1(x)
        q = n2(x)
        lo, hi = min(lo, q), max(hi, q)
    lam = np.linalg.eigvalsh(M)
    # on the sup-norm sphere, ||x||_2 is between 1 and sqrt(n), so
    # sqrt(lam_min) <= N2/N1 <= sqrt(n lam_max)
    return (lo >= np.sqrt(lam[0]) - 1e-9
            and hi <= np.sqrt(n * lam[-1]) + 1e-9), min(lo - np.sqrt(lam[0]),
                                                        np.sqrt(n * lam[-1]) - hi)


def t_projection(rng):
    """Thm 4.35: a unique closest point p in U, and v - p orthogonal to U."""
    n = int(rng.integers(2, 7))
    k = int(rng.integers(1, n))
    B = np.linalg.qr(rng.normal(size=(n, k)))[0]
    v = rng.normal(size=n)
    p = B @ (B.T @ v)
    if np.max(np.abs(B.T @ (v - p))) > 1e-10:
        return False, -1.0
    base = np.linalg.norm(v - p)
    best = base
    for _ in range(60):
        q = B @ rng.normal(size=k) * rng.uniform(0.1, 2)
        best = min(best, np.linalg.norm(v - (p + q)))
    return best >= base - 1e-12, best - base


def t_riesz(rng):
    """Thm 4.36: phi(v) = <u, v> with ||phi||_* = ||u||."""
    n = int(rng.integers(2, 7))
    u = rng.normal(size=n)
    phi = lambda v: u @ v
    sup = 0.0
    for _ in range(2000):
        v = rng.normal(size=n)
        sup = max(sup, abs(phi(v)) / np.linalg.norm(v))
    nu = np.linalg.norm(u)
    return sup <= nu + 1e-12 and sup > 0.5 * nu, nu - sup


run("Thm 4.17  equivalence of norms", t_equivalence)
run("Thm 4.35  projection on a complete subspace", t_projection)
run("Thm 4.36  Riesz representation", t_riesz)

ok   Thm 4.17  equivalence of norms                  200 trials, tightest margin 2.316e-04
ok   Thm 4.35  projection on a complete subspace     200 trials, tightest margin 0.000e+00


ok   Thm 4.36  Riesz representation                  200 trials, tightest margin 1.462e-10


## Chapter 5: the spectral theorem and the singular value decomposition

The two rate statements are the ones worth stressing. Theorem 5.11 claims
equality for a suitable $e_0$, so the margin should come out at
floating-point level — and it does. Proposition 5.18 is subtler than it
looks: the iteration leaves the component in $\ker A$ untouched, which is
exactly why the limit is the least-squares solution closest to $x_0$.

In [4]:
def t_sylvester(rng):
    """Thm 5.3: the signature is invariant under G -> P^T G P."""
    n = int(rng.integers(2, 7))
    Q = np.linalg.qr(rng.normal(size=(n, n)))[0]
    d = rng.choice([-2.0, -0.5, 0.5, 2.0], n)
    G = Q @ np.diag(d) @ Q.T
    G = (G + G.T) / 2
    sig0 = (int(np.sum(d > 1e-12)), int(np.sum(d < -1e-12)), int(np.sum(np.abs(d) <= 1e-12)))
    # a well-conditioned change of basis: the theorem is about any
    # invertible P, but a nearly singular one only tests floating point
    Q2 = np.linalg.qr(rng.normal(size=(n, n)))[0]
    P = Q2 @ np.diag(rng.uniform(0.5, 2.0, n)) @ np.linalg.qr(rng.normal(size=(n, n)))[0]
    H = P.T @ G @ P
    e = np.linalg.eigvalsh((H + H.T) / 2)
    tol = 1e-9 * max(1.0, np.max(np.abs(e)))
    sig1 = (int(np.sum(e > tol)), int(np.sum(e < -tol)), int(np.sum(np.abs(e) <= tol)))
    return sig0 == sig1, 0.0 if sig0 == sig1 else -1.0


def t_courant_fischer(rng):
    """Thm 5.9: lambda_k = max over dim U = k of min of <v, Sv>, ||v||=1."""
    n = int(rng.integers(2, 6))
    S = spd(rng, n, -3, 3)
    S = (S + S.T) / 2
    lam = np.sort(np.linalg.eigvalsh(S))[::-1]
    k = int(rng.integers(1, n + 1))
    # the eigenspace of the top k eigenvalues achieves the max
    evals, evecs = np.linalg.eigh(S)
    U = evecs[:, np.argsort(evals)[::-1][:k]]
    inner = U.T @ S @ U
    achieved = np.min(np.linalg.eigvalsh(inner))
    if abs(achieved - lam[k - 1]) > 1e-8:
        return False, -abs(achieved - lam[k - 1])
    # no random k-dimensional subspace beats it
    worst = 0.0
    for _ in range(30):
        V = np.linalg.qr(rng.normal(size=(n, k)))[0]
        val = np.min(np.linalg.eigvalsh(V.T @ S @ V))
        worst = max(worst, val - lam[k - 1])
    return worst <= 1e-8, -worst


def t_quadratic_rate(rng):
    """Thm 5.11: ||e_k|| <= r(eta)^k ||e_0||, r = max(|1-eta l1|, |1-eta ln|)."""
    n = int(rng.integers(2, 6))
    S = spd(rng, n)
    lam = np.sort(np.linalg.eigvalsh(S))[::-1]
    eta = float(rng.uniform(0.01, 2.0 / lam[0]))
    r = max(abs(1 - eta * lam[0]), abs(1 - eta * lam[-1]))
    e = rng.normal(size=n)
    e0 = np.linalg.norm(e)
    margin = np.inf
    for k in range(1, 12):
        e = (np.eye(n) - eta * S) @ e
        margin = min(margin, r ** k * e0 - np.linalg.norm(e))
    # optimal step and its rate
    eta_opt = 2 / (lam[0] + lam[-1])
    kappa = lam[0] / lam[-1]
    r_opt = max(abs(1 - eta_opt * lam[0]), abs(1 - eta_opt * lam[-1]))
    if abs(r_opt - (kappa - 1) / (kappa + 1)) > 1e-9:
        return False, -abs(r_opt - (kappa - 1) / (kappa + 1))
    if r_opt > r + 1e-12:
        return False, -(r_opt - r)
    return margin >= -1e-9 * e0, margin


run("Thm 5.3   Sylvester's law of inertia", t_sylvester)
run("Thm 5.9   Courant-Fischer", t_courant_fischer)
run("Thm 5.11  rate on a quadratic", t_quadratic_rate)

ok   Thm 5.3   Sylvester's law of inertia            200 trials, tightest margin 0.000e+00


ok   Thm 5.9   Courant-Fischer                       200 trials, tightest margin -3.997e-15
ok   Thm 5.11  rate on a quadratic                   200 trials, tightest margin 9.841e-18


In [5]:
def t_ls_gd(rng):
    """Prop 5.18: the limit is the least-squares solution closest to x0."""
    m, n = int(rng.integers(3, 7)), int(rng.integers(2, 6))
    r = int(min(m, n) - 1) or 1
    A = rng.normal(size=(m, r)) @ rng.normal(size=(r, n))
    b = rng.normal(size=m)
    x0 = rng.normal(size=n)
    sv = np.linalg.svd(A, compute_uv=False)
    sv = sv[sv > 1e-10]
    hi = 2.0 / sv[0] ** 2
    eta = float(rng.uniform(0.05 * hi, 0.95 * hi))
    rate = max(abs(1 - eta * sv[0] ** 2), abs(1 - eta * sv[-1] ** 2))
    # iterate long enough for the rate at hand, not a fixed budget
    steps = int(min(4e5, max(2e3, np.ceil(np.log(1e-12) / np.log(rate)))))
    x = x0.copy()
    for _ in range(steps):
        x = x - eta * (A.T @ (A @ x - b))
    # claimed limit: x0 + the min-norm correction in im(A^T)
    xinf = x0 + np.linalg.pinv(A) @ (b - A @ x0)
    if np.linalg.norm(x - xinf) > 1e-5 * (1 + np.linalg.norm(xinf)):
        return False, -np.linalg.norm(x - xinf)
    y, margin = x0.copy(), np.inf
    d0 = np.linalg.norm(x0 - xinf)
    for k in range(1, 15):
        y = y - eta * (A.T @ (A @ y - b))
        margin = min(margin, rate ** k * d0 - np.linalg.norm(y - xinf))
    return margin >= -1e-8 * (1 + d0), margin


def t_eckart_young(rng):
    """Thm 5.20: the truncated SVD is a best rank-k approximation."""
    m, n = int(rng.integers(3, 7)), int(rng.integers(3, 7))
    A = rng.normal(size=(m, n))
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    k = int(rng.integers(1, min(m, n)))
    Ak = (U[:, :k] * s[:k]) @ Vt[:k]
    e2 = np.linalg.norm(A - Ak, 2)                    # spectral
    eF = np.linalg.norm(A - Ak, "fro")
    if abs(e2 - s[k]) > 1e-8 or abs(eF - np.sqrt(np.sum(s[k:] ** 2))) > 1e-8:
        return False, -1.0
    best2, bestF = e2, eF
    for _ in range(60):
        B = rng.normal(size=(m, k)) @ rng.normal(size=(k, n))
        B = Ak + 0.3 * rng.normal() * (B / (np.linalg.norm(B) + 1e-12)) * np.linalg.norm(Ak)
        # project B back to rank k exactly
        Ub, sb, Vbt = np.linalg.svd(B, full_matrices=False)
        B = (Ub[:, :k] * sb[:k]) @ Vbt[:k]
        best2 = min(best2, np.linalg.norm(A - B, 2))
        bestF = min(bestF, np.linalg.norm(A - B, "fro"))
    return best2 >= e2 - 1e-9 and bestF >= eF - 1e-9, min(best2 - e2, bestF - eF)


def t_procrustes(rng):
    """Prop 5.21: the orthogonal Procrustes solution is U V^T of B A^T."""
    n = int(rng.integers(2, 6))
    k = int(rng.integers(2, 7))
    A = rng.normal(size=(n, k))
    B = rng.normal(size=(n, k))
    U, _, Vt = np.linalg.svd(B @ A.T)
    Q = U @ Vt
    base = np.linalg.norm(B - Q @ A, "fro")
    best = base
    for _ in range(80):
        R = np.linalg.qr(rng.normal(size=(n, n)))[0]
        if np.linalg.det(R) * np.linalg.det(Q) < 0:
            pass                                       # both orthogonal groups allowed
        best = min(best, np.linalg.norm(B - R @ A, "fro"))
    return best >= base - 1e-8, best - base


run("Prop 5.18 gradient descent for least squares", t_ls_gd, 40)
run("Thm 5.20  Eckart-Young-Mirsky", t_eckart_young)
run("Prop 5.21 orthogonal Procrustes", t_procrustes)

ok   Prop 5.18 gradient descent for least squares     40 trials, tightest margin -1.221e-15


ok   Thm 5.20  Eckart-Young-Mirsky                   200 trials, tightest margin 0.000e+00


ok   Prop 5.21 orthogonal Procrustes                 200 trials, tightest margin 0.000e+00


## Chapters 6 and 7: the differential, the chain rule, backpropagation

The two differentials of Chapter 6 are checked against a central difference.
Chapter 7's claim that copy and addition are dual is a statement about two
matrices being transposes of each other, which is checked exactly; the
backpropagation theorem is checked by comparing the reverse pass of a small
network with finite differences of its loss.

In [6]:
def t_dinverse(rng):
    """Prop 6.26: D(A -> A^-1)(A) H = -A^-1 H A^-1."""
    n = int(rng.integers(2, 6))
    A = rng.normal(size=(n, n)) + 2 * n * np.eye(n)
    if np.linalg.cond(A) > 50:          # a derivative test, not a conditioning test
        return True, 1.0
    H = rng.normal(size=(n, n))
    Ai = np.linalg.inv(A)
    claim = -Ai @ H @ Ai
    t = 1e-6
    num = (np.linalg.inv(A + t * H) - np.linalg.inv(A - t * H)) / (2 * t)
    err = np.max(np.abs(num - claim)) / max(1.0, np.max(np.abs(claim)))
    return err < 1e-6, -err


def t_ddet(rng):
    """Prop 6.27: d(det)(A)H = det(A) tr(A^-1 H), d(log det)(A)H = tr(A^-1 H)."""
    n = int(rng.integers(2, 6))
    A = rng.normal(size=(n, n)) + 2 * n * np.eye(n)
    if np.linalg.cond(A) > 50:          # a derivative test, not a conditioning test
        return True, 1.0
    H = rng.normal(size=(n, n))
    d = np.linalg.det(A)
    claim = d * np.trace(np.linalg.solve(A, H))
    t = 1e-6
    num = (np.linalg.det(A + t * H) - np.linalg.det(A - t * H)) / (2 * t)
    e1 = abs(num - claim) / max(1.0, abs(claim))
    if d <= 0:
        return e1 < 1e-6, -e1
    claim2 = np.trace(np.linalg.solve(A, H))
    num2 = (np.log(np.linalg.det(A + t * H))
            - np.log(np.linalg.det(A - t * H))) / (2 * t)
    e2 = abs(num2 - claim2) / max(1.0, abs(claim2))
    return max(e1, e2) < 1e-6, -max(e1, e2)


def t_copy_add_dual(rng):
    """Lemma 7.11, Prop 7.12: the dual of the copy map is addition, and
    the dual of addition is the copy map."""
    n = int(rng.integers(1, 6))
    copy = np.vstack([np.eye(n), np.eye(n)])             # v -> (v, v)
    add = np.hstack([np.eye(n), np.eye(n)])              # (a, b) -> a + b
    if np.max(np.abs(copy.T - add)) > 0:
        return False, -1.0
    v = rng.normal(size=n)
    ab = rng.normal(size=2 * n)
    lhs = (copy @ v) @ ab
    rhs = v @ (add @ ab)
    return abs(lhs - rhs) < 1e-12 * (1 + abs(lhs)), -abs(lhs - rhs)


def t_backprop(rng):
    """Thm 7.10: for L = l(f_k(...f_1(x))), the differential of the loss is
    the composition of the differentials, and the gradient is obtained by
    applying the adjoints in reverse order."""
    n = int(rng.integers(2, 5))
    depth = int(rng.integers(2, 5))
    Ws = [rng.normal(size=(n, n)) / np.sqrt(n) for _ in range(depth)]
    bs = [rng.normal(size=n) * 0.1 for _ in range(depth)]
    act = np.tanh
    dact = lambda z: 1 - np.tanh(z) ** 2
    c = rng.normal(size=n)

    def forward(x):
        zs, a = [], x
        for W, b in zip(Ws, bs):
            z = W @ a + b
            zs.append((a, z))
            a = act(z)
        return c @ a, zs, a

    x = rng.normal(size=n)
    L, zs, a = forward(x)
    # reverse pass: adjoints in reverse order
    g = c.copy()
    for (a_in, z), W in zip(reversed(zs), reversed(Ws)):
        g = W.T @ (dact(z) * g)
    # finite differences
    t = 1e-6
    num = np.array([(forward(x + t * np.eye(n)[i])[0] - L) / t for i in range(n)])
    err = np.max(np.abs(num - g)) / max(1.0, np.max(np.abs(g)))
    return err < 1e-4, -err


run("Prop 6.26 the differential of the inverse", t_dinverse)
run("Prop 6.27 the differential of the determinant", t_ddet)
run("Lem 7.11  copy and addition are dual", t_copy_add_dual)
run("Thm 7.10  backpropagation", t_backprop)

ok   Prop 6.26 the differential of the inverse       200 trials, tightest margin -8.886e-11
ok   Prop 6.27 the differential of the determinant   200 trials, tightest margin -2.293e-07
ok   Lem 7.11  copy and addition are dual            200 trials, tightest margin -1.776e-15
ok   Thm 7.10  backpropagation                       200 trials, tightest margin -2.432e-06


## Chapters 8 and 9: second order, convergence rates, Newton

Theorem 8.9 is tested in its mean-value form: the $\theta\in(0,1)$ whose
existence it asserts is located by bisection, and the second form is tested
by watching $r(h)/\lVert h\rVert^2$ go to zero. Theorem 8.14 is tested in
both directions at once — a convex and a non-convex family — so that an
implication stated the wrong way round would fail.

Theorem 9.12 is the one where the constant matters: the claim is
$\lVert e_{k+1}\rVert\le\beta L\lVert e_k\rVert^2$ with
$\beta=\lVert DF(a^*)^{-1}\rVert$, on a ball of radius $1/(2\beta L)$, and
both that and the doubly exponential bound are checked at every step.

In [7]:
def t_hessian_symmetry(rng):
    """Thm 8.6: D^2 f(a) is symmetric for a C^2 function."""
    n = int(rng.integers(2, 5))
    A = rng.normal(size=(n, n))
    c = rng.normal(size=n)
    f = lambda x: np.sin(c @ x) + np.exp(-(x @ A @ x) / (2 * n)) + np.sum(x ** 4) / 12
    a = rng.normal(size=n) * 0.5
    t = 1e-4
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei, ej = np.eye(n)[i], np.eye(n)[j]
            H[i, j] = (f(a + t * ei + t * ej) - f(a + t * ei - t * ej)
                       - f(a - t * ei + t * ej) + f(a - t * ei - t * ej)) / (4 * t * t)
    err = np.max(np.abs(H - H.T)) / max(1.0, np.max(np.abs(H)))
    return err < 1e-4, -err


def t_taylor(rng):
    """Thm 8.9: the mean-value form has a theta in (0,1), and the
    remainder is o(||h||^2)."""
    n = int(rng.integers(1, 4))
    c = rng.normal(size=n)
    A = spd(rng, n)
    f = lambda x: np.sin(c @ x) + (x @ A @ x) / 2 + np.sum(x ** 3) / 6
    df = lambda x: np.cos(c @ x) * c + A @ x + x ** 2 / 2
    def d2f(x):
        return -np.sin(c @ x) * np.outer(c, c) + A + np.diag(x)
    a = rng.normal(size=n) * 0.4
    h = rng.normal(size=n) * 0.3
    target = f(a + h) - f(a) - df(a) @ h
    g = lambda th: 0.5 * h @ d2f(a + th * h) @ h - target
    lo, hi = 0.0, 1.0
    if g(lo) * g(hi) > 0:
        return True, 1.0                      # no sign change sampled: skip
    for _ in range(80):
        mid = (lo + hi) / 2
        if g(lo) * g(mid) <= 0:
            hi = mid
        else:
            lo = mid
    theta = (lo + hi) / 2
    if not (0 < theta < 1) or abs(g(theta)) > 1e-10 * (1 + abs(target)):
        return False, -abs(g(theta))
    # the o(||h||^2) form
    ratios = []
    for s in (1e-1, 1e-2, 1e-3):
        hs = h / np.linalg.norm(h) * s
        r = f(a + hs) - f(a) - df(a) @ hs - 0.5 * hs @ d2f(a) @ hs
        ratios.append(abs(r) / s ** 2)
    return ratios[0] > ratios[1] > ratios[2] - 1e-12, ratios[1] - ratios[2]


def t_convexity(rng):
    """Thm 8.14: for C^2 f, D^2 f >= 0 everywhere iff f lies above its
    tangent planes."""
    n = int(rng.integers(1, 4))
    A = spd(rng, n)
    conv = bool(rng.integers(0, 2))
    if conv:
        f = lambda x: (x @ A @ x) / 2 + np.sum(np.exp(x / 3))
        df = lambda x: A @ x + np.exp(x / 3) / 3
    else:
        f = lambda x: (x @ A @ x) / 2 - 3 * np.sum(x ** 2)
        df = lambda x: A @ x - 6 * x
    hess_psd = True
    for _ in range(60):
        x = rng.normal(size=n) * 2
        t = 1e-4
        H = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                ei, ej = np.eye(n)[i], np.eye(n)[j]
                H[i, j] = (f(x + t * ei + t * ej) - f(x + t * ei - t * ej)
                           - f(x - t * ei + t * ej) + f(x - t * ei - t * ej)) / (4 * t * t)
        if np.min(np.linalg.eigvalsh((H + H.T) / 2)) < -1e-4:
            hess_psd = False
            break
    above = True
    for _ in range(200):
        x, y = rng.normal(size=n) * 2, rng.normal(size=n) * 2
        if f(y) < f(x) + df(x) @ (y - x) - 1e-7:
            above = False
            break
    return hess_psd == above, 0.0 if hess_psd == above else -1.0


run("Thm 8.6   symmetry of the second differential", t_hessian_symmetry, 40)
run("Thm 8.9   Taylor of order two", t_taylor)
run("Thm 8.14  characterizations of convexity", t_convexity, 40)

ok   Thm 8.6   symmetry of the second differential    40 trials, tightest margin -0.000e+00


ok   Thm 8.9   Taylor of order two                   200 trials, tightest margin 3.778e-07


ok   Thm 8.14  characterizations of convexity         40 trials, tightest margin 0.000e+00


In [8]:
def t_gd_rates(rng):
    """Thm 8.19: step 1/L gives (1 - mu/L)^k on f - f*, and the 2L/K bound."""
    n = int(rng.integers(2, 6))
    f, g, xs, mu, L = quad_problem(rng, n)
    x = rng.normal(size=n) * 2
    f0, fs = f(x), f(xs)
    gaps, grads = [], []
    for _ in range(60):
        grads.append(np.linalg.norm(g(x)) ** 2)
        x = x - g(x) / L
        gaps.append(f(x) - fs)
    m1 = min((1 - mu / L) ** (k + 1) * (f0 - fs) - gap for k, gap in enumerate(gaps))
    K = len(grads)
    m2 = 2 * L * (f0 - fs) / K - min(grads)
    return min(m1, m2) >= -1e-9 * (1 + f0 - fs), min(m1, m2)


def t_gd_contraction(rng):
    """Thm 9.9: g(a) = a - eta grad f is Lipschitz with max(|1-eta mu|,|1-eta L|)."""
    n = int(rng.integers(2, 6))
    f, g, xs, mu, L = quad_problem(rng, n)
    eta = float(rng.uniform(0.01, 2.0 / L))
    k = max(abs(1 - eta * mu), abs(1 - eta * L))
    if not k < 1:
        return False, -1.0
    margin = np.inf
    for _ in range(20):
        a, b = rng.normal(size=n), rng.normal(size=n)
        ga, gb = a - eta * g(a), b - eta * g(b)
        margin = min(margin, k * np.linalg.norm(a - b) - np.linalg.norm(ga - gb))
    x = rng.normal(size=n) * 3
    d0 = np.linalg.norm(x - xs)
    for j in range(1, 30):
        x = x - eta * g(x)
        margin = min(margin, k ** j * d0 - np.linalg.norm(x - xs))
    eta_opt = 2 / (L + mu)
    if abs(max(abs(1 - eta_opt * mu), abs(1 - eta_opt * L)) - (L - mu) / (L + mu)) > 1e-12:
        return False, -1.0
    return margin >= -1e-9 * (1 + d0), margin


run("Thm 8.19  two rates for gradient descent", t_gd_rates)
run("Thm 9.9   gradient descent is a contraction", t_gd_contraction)

ok   Thm 8.19  two rates for gradient descent        200 trials, tightest margin -8.882e-16
ok   Thm 9.9   gradient descent is a contraction     200 trials, tightest margin -4.475e-16


In [9]:
def t_newton_quadratic(rng):
    """Thm 9.12: ||e_{k+1}|| <= beta L ||e_k||^2 and ||e_k|| <= (1/beta L)(1/2)^{2^k}."""
    n = int(rng.integers(1, 4))
    A = rng.normal(size=(n, n)) + 2 * np.eye(n)
    c = rng.normal(size=n) * 0.3
    # F(x) = A x + c * (x.x) : F' Lipschitz, F(xstar) = 0 by construction
    xs = rng.normal(size=n) * 0.1
    d = A @ xs + c * (xs @ xs)
    F = lambda x: A @ x + c * (x @ x) - d
    DF = lambda x: A + 2 * np.outer(c, x)
    beta = np.linalg.norm(np.linalg.inv(DF(xs)), 2)
    L = 2 * np.linalg.norm(c) * np.sqrt(n) * 2          # a Lipschitz constant for DF
    r = 1 / (2 * beta * L)
    e0 = rng.normal(size=n)
    e0 = e0 / np.linalg.norm(e0) * rng.uniform(0, r)
    x = xs + e0
    margin = np.inf
    for k in range(6):
        ek = np.linalg.norm(x - xs)
        x = x - np.linalg.solve(DF(x), F(x))
        margin = min(margin, beta * L * ek ** 2 - np.linalg.norm(x - xs))
        margin = min(margin, (1 / (beta * L)) * 0.5 ** (2 ** k) - ek)
        if ek < 1e-13:
            break
    return margin >= -1e-10, margin


def t_newton_invariance(rng):
    """Prop 9.13: N_G = phi^-1 . N_F . phi for G = B . F . phi."""
    n = int(rng.integers(2, 5))
    A = rng.normal(size=(n, n)) + 2 * np.eye(n)
    c = rng.normal(size=n) * 0.2
    F = lambda x: A @ x + c * (x @ x)
    DF = lambda x: A + 2 * np.outer(c, x)
    P = rng.normal(size=(n, n)) + 2 * np.eye(n)
    q = rng.normal(size=n)
    B = rng.normal(size=(n, n)) + 2 * np.eye(n)
    phi = lambda y: P @ y + q
    G = lambda y: B @ F(phi(y))
    DG = lambda y: B @ DF(phi(y)) @ P
    y = rng.normal(size=n) * 0.2
    NG = y - np.linalg.solve(DG(y), G(y))
    x = phi(y)
    NF = x - np.linalg.solve(DF(x), F(x))
    lhs = phi(NG)
    return np.allclose(lhs, NF, atol=1e-8), -float(np.max(np.abs(lhs - NF)))


def t_woodbury(rng):
    """Prop 9.17: (A + UCV)^-1 = A^-1 - A^-1 U (I + C V A^-1 U)^-1 C V A^-1."""
    n, k = int(rng.integers(2, 7)), int(rng.integers(1, 4))
    A = rng.normal(size=(n, n)) + n * np.eye(n)
    U = rng.normal(size=(n, k))
    V = rng.normal(size=(k, n))
    C = rng.normal(size=(k, k)) + k * np.eye(k)
    Ai = np.linalg.inv(A)
    lhs = np.linalg.inv(A + U @ C @ V)
    mid = np.linalg.inv(np.eye(k) + C @ V @ Ai @ U)
    rhs = Ai - Ai @ U @ mid @ C @ V @ Ai
    err = np.max(np.abs(lhs - rhs))
    return err <= 1e-7 * max(1.0, np.max(np.abs(lhs))), -err


run("Thm 9.12  local quadratic convergence", t_newton_quadratic)
run("Prop 9.13 affine invariance of Newton", t_newton_invariance)
run("Prop 9.17 low-rank updates", t_woodbury)

ok   Thm 9.12  local quadratic convergence           200 trials, tightest margin -3.171e-16
ok   Prop 9.13 affine invariance of Newton           200 trials, tightest margin -4.846e-12
ok   Prop 9.17 low-rank updates                      200 trials, tightest margin -3.803e-10


## Chapters 10 and 11: inverse and implicit functions, constrained optimization

The two theorems of Chapter 10 are existence statements, so what is tested
is their conclusion: the local inverse is produced by Newton's method, and
its derivative is compared with $DF(a)^{-1}$; the implicit function's
derivative is compared with $-(\partial_yg)^{-1}\partial_xg$.

The sign of the multipliers in Theorem 11.7 and the sign in Proposition 11.4
are the two places in Chapter 11 where a convention can silently flip, so
both are checked against a problem whose solution is known in closed form.

In [10]:
def t_inverse_function(rng):
    """Thm 10.2: a local inverse exists near a with DF(a) invertible, and
    D(F^-1)(F(a)) = DF(a)^-1."""
    n = int(rng.integers(1, 4))
    A = rng.normal(size=(n, n)) + 2 * n * np.eye(n)
    c = rng.normal(size=n) * 0.2
    F = lambda x: A @ x + c * np.sin(x @ x)
    DF = lambda x: A + 2 * np.outer(c * np.cos(x @ x), x)
    a = rng.normal(size=n) * 0.3
    b = F(a)
    J = DF(a)
    if abs(np.linalg.det(J)) < 1e-8:
        return True, 1.0
    # solve F(x) = b + t w by Newton, for a small t
    w = rng.normal(size=n)
    t = 1e-5
    xs = []
    for sgn in (+1, -1):
        x = a.copy()
        for _ in range(80):
            x = x - np.linalg.solve(DF(x), F(x) - (b + sgn * t * w))
        if np.linalg.norm(F(x) - (b + sgn * t * w)) > 1e-10 * (1 + np.linalg.norm(b)):
            return False, -1.0
        xs.append(x)
    num = (xs[0] - xs[1]) / (2 * t)              # central difference
    claim = np.linalg.solve(J, w)
    err = np.max(np.abs(num - claim)) / max(1.0, np.max(np.abs(claim)))
    return err < 1e-6, -err


def t_implicit_function(rng):
    """Thm 10.5: g(x, y) = 0 defines y = h(x) with
    Dh(x) = -(d_y g)^-1 (d_x g)."""
    p, q = int(rng.integers(1, 4)), int(rng.integers(1, 4))
    Ax = rng.normal(size=(q, p))
    Ay = rng.normal(size=(q, q)) + 2 * q * np.eye(q)
    c = rng.normal(size=q) * 0.1
    g = lambda x, y: Ax @ x + Ay @ y + c * np.sin(y @ y)
    dgx = lambda x, y: Ax
    dgy = lambda x, y: Ay + 2 * np.outer(c * np.cos(y @ y), y)
    x0 = rng.normal(size=p) * 0.2
    y0 = np.zeros(q)
    for _ in range(80):
        y0 = y0 - np.linalg.solve(dgy(x0, y0), g(x0, y0))
    if np.linalg.norm(g(x0, y0)) > 1e-10:
        return True, 1.0
    claim = -np.linalg.solve(dgy(x0, y0), dgx(x0, y0))
    t = 1e-6
    u = rng.normal(size=p)
    y1 = y0.copy()
    for _ in range(80):
        y1 = y1 - np.linalg.solve(dgy(x0 + t * u, y1), g(x0 + t * u, y1))
    num = (y1 - y0) / t
    err = np.max(np.abs(num - claim @ u)) / max(1.0, np.max(np.abs(claim @ u)))
    return err < 1e-3, -err


run("Thm 10.2  inverse function theorem", t_inverse_function)
run("Thm 10.5  implicit function theorem", t_implicit_function)

ok   Thm 10.2  inverse function theorem              200 trials, tightest margin -6.278e-07


ok   Thm 10.5  implicit function theorem             200 trials, tightest margin -4.551e-07


In [11]:
def t_kkt_signs(rng):
    """Thm 11.7: at a minimum, mu_j >= 0 and mu_j h_j(a) = 0.

    Minimize f(x) = |x - z|^2/2 on the box h_j(x) = x_j - 1 <= 0,
    -x_j <= 0, whose solution is the clipping of z.  The convention is
    df(a) + sum mu_j dh_j(a) = 0 with mu_j >= 0.
    """
    n = int(rng.integers(1, 5))
    z = rng.normal(size=n) * 2
    a = np.clip(z, 0.0, 1.0)
    grad = a - z                                        # df(a)
    # active constraints: x_j <= 1 (gradient e_j), -x_j <= 0 (gradient -e_j)
    mus = []
    for j in range(n):
        if a[j] >= 1 - 1e-12 and z[j] > 1:
            mu = -grad[j]                               # df_j + mu * (+1) = 0
            mus.append((mu, 0.0))
        elif a[j] <= 1e-12 and z[j] < 0:
            mu = grad[j]                                # df_j + mu * (-1) = 0
            mus.append((mu, 0.0))
        else:
            if abs(grad[j]) > 1e-9:
                return False, -abs(grad[j])             # interior: must be stationary
            mus.append((0.0, 0.0))
    worst = min((mu for mu, _ in mus), default=1.0)
    return worst >= -1e-12, worst


def t_sensitivity(rng):
    """Prop 11.4: dv(c) = -lambda(c) with df(a) + lambda . Dg(a) = 0.

    f(x) = |x|^2/2 on g(x) = u.x = c, so a = c u/|u|^2 and v(c) = c^2/(2|u|^2).
    """
    n = int(rng.integers(2, 6))
    u = rng.normal(size=n)
    c = float(rng.normal())
    a = c * u / (u @ u)
    # stationarity: a + lam * u = 0  ->  lam = -c/|u|^2
    lam = -c / (u @ u)
    dv = c / (u @ u)                                    # d/dc of c^2/(2|u|^2)
    return abs(dv + lam) <= 1e-12 * (1 + abs(dv)), -abs(dv + lam)


def t_pgd(rng):
    """Thm 11.11: projected gradient descent on a box, same rate."""
    n = int(rng.integers(2, 6))
    f, g, _, mu, L = quad_problem(rng, n)
    eta = float(rng.uniform(0.01, 2.0 / L))
    k = max(abs(1 - eta * mu), abs(1 - eta * L))
    P = lambda x: np.clip(x, -1.0, 1.0)
    x = P(rng.normal(size=n) * 3)
    for _ in range(4000):
        x = P(x - eta * g(x))
    xstar = x.copy()
    # xstar minimizes f on the box: no feasible direction decreases f
    for _ in range(40):
        y = P(xstar + rng.normal(size=n) * rng.uniform(0.01, 0.5))
        if f(y) < f(xstar) - 1e-9:
            return False, f(y) - f(xstar)
    y = P(rng.normal(size=n) * 3)
    d0 = np.linalg.norm(y - xstar)
    margin = np.inf
    for j in range(1, 25):
        y = P(y - eta * g(y))
        margin = min(margin, k ** j * d0 - np.linalg.norm(y - xstar))
    return margin >= -1e-8 * (1 + d0), margin


run("Thm 11.7  KKT signs and slackness", t_kkt_signs)
run("Prop 11.4 multipliers are sensitivities", t_sensitivity)
run("Thm 11.11 projected gradient descent", t_pgd, 40)

ok   Thm 11.7  KKT signs and slackness               200 trials, tightest margin 0.000e+00
ok   Prop 11.4 multipliers are sensitivities         200 trials, tightest margin -0.000e+00


ok   Thm 11.11 projected gradient descent             40 trials, tightest margin 3.359e-16


## Chapter 12: submanifolds

Proposition 12.7 is the only item here whose uniqueness half can be tested:
a subspace of the right dimension that is tilted away from $T_pM$ is made to
fail the contact condition, by approaching $p$ along the tangent direction
it misses. The dimension counts of Theorems 12.11 and 12.13 are read off the
rank of the relevant linear map rather than assumed.

In [12]:
def t_normal_projection(rng):
    """Prop 12.8: Pi_p = id - A*(A A*)^-1 A with A = Dg(p) is the
    orthogonal projection on the tangent space ker A."""
    n = int(rng.integers(2, 7))
    m = int(rng.integers(1, n))
    A = rng.normal(size=(m, n))
    if np.linalg.matrix_rank(A) < m:
        return True, 1.0
    Pi = np.eye(n) - A.T @ np.linalg.solve(A @ A.T, A)
    if np.max(np.abs(Pi @ Pi - Pi)) > 1e-9 or np.max(np.abs(Pi - Pi.T)) > 1e-9:
        return False, -1.0
    v = rng.normal(size=n)
    if np.max(np.abs(A @ (Pi @ v))) > 1e-9 * (1 + np.linalg.norm(v)):
        return False, -1.0
    # Pi v is the point of ker A closest to v
    base = np.linalg.norm(v - Pi @ v)
    best = base
    ns = np.linalg.svd(A)[2][m:]
    for _ in range(60):
        z = ns.T @ rng.normal(size=ns.shape[0])
        for s in (-1.0, -0.2, 0.2, 1.0):
            best = min(best, np.linalg.norm(v - (Pi @ v + s * z)))
    return best >= base - 1e-9, best - base


def t_first_order_contact(rng):
    """Prop 12.7: dist(x, p + T_p M) = o(||x - p||) on the sphere, and a
    tilted subspace of the same dimension fails it."""
    n = int(rng.integers(2, 6))
    p = rng.normal(size=n)
    p = p / np.linalg.norm(p)
    T = np.linalg.svd(p.reshape(1, -1))[2][1:]            # tangent space at p
    ratios = []
    for s in (1e-1, 1e-2, 1e-3):
        u = T.T @ rng.normal(size=n - 1)
        u = u / np.linalg.norm(u)
        x = p + s * u
        x = x / np.linalg.norm(x)                         # back on the sphere
        d = x - p
        dist = np.linalg.norm(d - T.T @ (T @ d))          # distance to p + T
        ratios.append(dist / np.linalg.norm(d))
    if not (ratios[0] > ratios[1] > ratios[2]):
        return False, -1.0
    # a tilted subspace of the same dimension does not have the property:
    # approach p along a tangent direction that the tilted space misses
    Tt = T.copy()
    Tt[0] = Tt[0] + 0.3 * p
    Tt = np.linalg.qr(Tt.T)[0].T
    # the direction of T that is farthest from Tt
    R = T @ (np.eye(n) - Tt.T @ Tt) @ T.T
    w = np.linalg.eigh((R + R.T) / 2)[1][:, -1]
    u = T.T @ w
    nu = np.linalg.norm(u)
    if nu < 1e-12:
        return True, 1.0
    u = u / nu
    if np.linalg.norm(u - Tt.T @ (Tt @ u)) < 1e-6:
        return True, 1.0                      # T is inside Tt: nothing to test
    bad = []
    for s in (1e-2, 1e-3, 1e-4):
        x = p + s * u
        x = x / np.linalg.norm(x)
        d = x - p
        bad.append(np.linalg.norm(d - Tt.T @ (Tt @ d)) / np.linalg.norm(d))
    # that ratio stays bounded away from 0 while the one for T goes to 0
    return min(bad) > 10 * ratios[2], min(bad) - 10 * ratios[2]


def t_submanifold_gradient(rng):
    """Prop 12.19: the gradient of f restricted to M is Pi_p grad f(p)."""
    n = int(rng.integers(2, 6))
    p = rng.normal(size=n)
    p = p / np.linalg.norm(p)
    M = spd(rng, n)
    c = rng.normal(size=n)
    f = lambda x: (x @ M @ x) / 2 + c @ x
    grad = lambda x: M @ x + c
    T = np.linalg.svd(p.reshape(1, -1))[2][1:]
    g_tan = T.T @ (T @ grad(p))
    # compare with the derivative along curves in the sphere
    err = 0.0
    for _ in range(20):
        v = T.T @ rng.normal(size=n - 1)
        t = 1e-6
        x = (p + t * v)
        x = x / np.linalg.norm(x)
        num = (f(x) - f(p)) / t
        err = max(err, abs(num - g_tan @ v) / max(1.0, abs(g_tan @ v)))
    return err < 1e-3, -err


run("Prop 12.8 the projection on the tangent space", t_normal_projection)
run("Prop 12.7 first-order contact", t_first_order_contact)
run("Prop 12.19 the gradient on a submanifold", t_submanifold_gradient)

ok   Prop 12.8 the projection on the tangent space   200 trials, tightest margin 0.000e+00
ok   Prop 12.7 first-order contact                   200 trials, tightest margin 2.776e-01
ok   Prop 12.19 the gradient on a submanifold        200 trials, tightest margin -1.552e-05


In [13]:
def t_stiefel_tangent(rng):
    """Thm 12.11: T_X St(n,k) = {V : X^T V + V^T X = 0}, of dimension nk - k(k+1)/2."""
    n, k = int(rng.integers(2, 7)), None
    k = int(rng.integers(1, n + 1))
    X = np.linalg.qr(rng.normal(size=(n, k)))[0]
    # a curve in the Stiefel manifold through X
    Ssym = rng.normal(size=(k, k)); Ssym = Ssym - Ssym.T          # skew
    Bperp = rng.normal(size=(n, k))
    Bperp = Bperp - X @ (X.T @ Bperp)
    V = X @ Ssym + Bperp
    t = 1e-6
    Y = np.linalg.qr(X + t * V)[0]
    # sign-fix the QR so the curve is continuous at t = 0
    sgn = np.sign(np.diag((X.T @ Y)))
    sgn[sgn == 0] = 1
    Y = Y * sgn
    num = (Y - X) / t
    if np.max(np.abs(X.T @ V + V.T @ X)) > 1e-9:
        return False, -1.0
    dim = n * k - k * (k + 1) // 2
    # rank of the linear map V -> X^T V + V^T X on R^{n x k}
    rows = []
    for i in range(n):
        for j in range(k):
            E = np.zeros((n, k)); E[i, j] = 1.0
            rows.append((X.T @ E + E.T @ X).ravel())
    ker = n * k - np.linalg.matrix_rank(np.array(rows))
    if ker != dim:
        return False, -1.0
    err = np.max(np.abs(num - V)) / max(1.0, np.max(np.abs(V)))
    return err < 5e-3, -err


def t_fixed_rank(rng):
    """Thm 12.13: the n x m matrices of rank exactly r form a submanifold of
    dimension r(n + m - r), with tangent space {U Y^T + Z V^T} at
    A = U S V^T."""
    n, m = int(rng.integers(2, 6)), int(rng.integers(2, 6))
    r = int(rng.integers(1, min(n, m) + 1))
    U = np.linalg.qr(rng.normal(size=(n, r)))[0]
    V = np.linalg.qr(rng.normal(size=(m, r)))[0]
    S = np.diag(rng.uniform(0.5, 2.0, r))
    A = U @ S @ V.T
    expect = r * (n + m - r)

    # (a) the velocities of the parametrization span a space of that dimension
    vel = []
    for _ in range(4 * expect + 20):
        dU = rng.normal(size=(n, r)); dV = rng.normal(size=(m, r))
        dS = rng.normal(size=(r, r))
        vel.append((dU @ S @ V.T + U @ dS @ V.T + U @ S @ dV.T).ravel())
    sv = np.linalg.svd(np.array(vel), compute_uv=False)
    dim = int(np.sum(sv > 1e-10 * sv[0]))
    if dim != expect:
        return False, -1.0

    # (b) each such velocity is of the form U Y^T + Z V^T, i.e. is killed
    #     by the two-sided projection on the orthogonal complements
    P = np.eye(n) - U @ U.T
    Q = np.eye(m) - V @ V.T
    T = np.array(vel[0]).reshape(n, m)
    if np.max(np.abs(P @ T @ Q)) > 1e-9 * max(1.0, np.max(np.abs(T))):
        return False, -1.0

    # (c) each velocity is the derivative of a curve of rank exactly r
    dU = rng.normal(size=(n, r)); dV = rng.normal(size=(m, r))
    dS = rng.normal(size=(r, r))
    Tex = dU @ S @ V.T + U @ dS @ V.T + U @ S @ dV.T
    t = 1e-6
    At = (U + t * dU) @ (S + t * dS) @ (V + t * dV).T
    if np.linalg.matrix_rank(At, tol=1e-8 * np.max(np.abs(At))) != r:
        return True, 1.0
    err = np.max(np.abs((At - A) / t - Tex)) / max(1.0, np.max(np.abs(Tex)))
    return err < 1e-3, -err


run("Thm 12.11 Stiefel tangent space", t_stiefel_tangent)
run("Thm 12.13 matrices of fixed rank", t_fixed_rank)

ok   Thm 12.11 Stiefel tangent space                 200 trials, tightest margin -4.697e-06


ok   Thm 12.13 matrices of fixed rank                200 trials, tightest margin -4.226e-06


## Chapters 14 and 15: Riemannian gradients and the Fisher metric

Proposition 14.6 says that $G^{-1}[\dd f]^\top$ transforms as a tangent
vector; the test builds a random change of chart and compares the two
columns, and separately checks that the direction of steepest ascent is the
normalized gradient. Example 14.4 is checked against $2\operatorname{artanh}
\lvert x\rvert$ by quadrature along the radius.

For Theorem 15.6 the model is the categorical one, where everything is
available in closed form: the Fisher matrix is $\operatorname{diag}(p) -
pp^\top$, which is singular exactly in the direction that kills every score
— the case part (1) describes. Part (2) is checked on a Gaussian whose mean
depends linearly on the parameters, so that the Fisher matrix in the new
chart can be computed from the model rather than from the transformation
rule.

In [14]:
def t_grad_chart(rng):
    """Prop 14.6: [grad f] = G^-1 [df]^T, and the column transforms as a
    tangent vector under a change of chart."""
    n = int(rng.integers(2, 6))
    G = spd(rng, n)
    df = rng.normal(size=n)                       # the row [df]
    g = np.linalg.solve(G, df)
    J = invertible(rng, n)                        # Jacobian of the chart change
    Gp = np.linalg.inv(J).T @ G @ np.linalg.inv(J)
    dfp = df @ np.linalg.inv(J)
    gp = np.linalg.solve(Gp, dfp)
    e1 = np.max(np.abs(gp - J @ g)) / max(1.0, np.max(np.abs(J @ g)))
    # steepest ascent among the unit vectors of the metric
    nrm = np.sqrt(g @ G @ g)
    best = df @ (g / nrm)
    worst = 0.0
    for _ in range(200):
        v = rng.normal(size=n)
        v = v / np.sqrt(v @ G @ v)
        worst = max(worst, df @ v - best)
    if abs(best - nrm) > 1e-9 * max(1.0, nrm):
        return False, -abs(best - nrm)
    return e1 < 1e-8 and worst < 1e-9, min(-e1, -worst)


def t_hyperbolic_distance(rng):
    """Example 14.4: in the Poincare ball with metric 4/(1-|x|^2)^2 times the
    Euclidean one, the distance from 0 to x is 2 artanh|x|."""
    n = int(rng.integers(1, 5))
    x = rng.normal(size=n)
    x = x / np.linalg.norm(x) * rng.uniform(0.05, 0.9)
    r = np.linalg.norm(x)
    # length of the radial segment, by quadrature
    nodes, weights = np.polynomial.legendre.leggauss(400)
    s = (nodes + 1) / 2 * r
    w = weights / 2 * r
    speed = 2 / (1 - s ** 2)
    length = float(np.sum(w * speed))
    claim = 2 * np.arctanh(r)
    err = abs(length - claim) / max(1.0, abs(claim))
    return err < 1e-10, -err


run("Prop 14.6  the gradient in a chart", t_grad_chart)
run("Ex 14.4    distance in the Poincare ball", t_hyperbolic_distance, 100)

ok   Prop 14.6  the gradient in a chart              200 trials, tightest margin -3.654e-14


ok   Ex 14.4    distance in the Poincare ball        100 trials, tightest margin -2.142e-14


In [15]:
def t_fisher_hessian(rng):
    """Thm 15.6(3): F_ij = -E[d_i d_j l], on the categorical model."""
    k = int(rng.integers(2, 7))
    th = rng.normal(size=k)
    p = np.exp(th - th.max())
    p = p / p.sum()
    F = np.diag(p) - np.outer(p, p)               # E[score score^T]
    # the second derivative of l_y = th_y - logsumexp(th) does not depend on y
    H = -(np.diag(p) - np.outer(p, p))
    e1 = np.max(np.abs(F + H))
    # F from the definition, by the exact expectation over y
    Fdef = np.zeros((k, k))
    for y in range(k):
        s = -p.copy()
        s[y] += 1.0
        Fdef += p[y] * np.outer(s, s)
    e2 = np.max(np.abs(Fdef - F))
    # symmetric, positive semidefinite, and singular exactly in the
    # direction that kills every score (the all-ones vector)
    lam = np.linalg.eigvalsh(F)
    if np.min(lam) < -1e-12 or np.max(np.abs(F - F.T)) > 1e-14:
        return False, -1.0
    one = np.ones(k) / np.sqrt(k)
    if np.max(np.abs(F @ one)) > 1e-12:
        return False, -1.0
    return max(e1, e2) < 1e-12, -max(e1, e2)


def t_fisher_chart(rng):
    """Thm 15.6(2): under a change of chart with Jacobian J the Fisher
    matrix becomes J^-T F J^-1.  Model: y ~ N(M theta + c, I)."""
    d = int(rng.integers(1, 5))
    M = invertible(rng, d)
    A = invertible(rng, d)                        # the new chart: th' = A th
    F = M.T @ M                                   # Fisher in the chart theta
    Mp = M @ np.linalg.inv(A)                     # mean as a function of th'
    Fp = Mp.T @ Mp                                # Fisher in the chart th'
    claim = np.linalg.inv(A).T @ F @ np.linalg.inv(A)
    err = np.max(np.abs(Fp - claim)) / max(1.0, np.max(np.abs(claim)))
    # and definite exactly when no direction kills every score
    pd = np.min(np.linalg.eigvalsh(F)) > 1e-10
    kills = np.min(np.linalg.svd(M, compute_uv=False)) < 1e-10
    return err < 1e-10 and pd == (not kills), -err


run("Thm 15.6(3) Fisher as minus the expected Hessian", t_fisher_hessian)
run("Thm 15.6(2) Fisher under a change of chart", t_fisher_chart)

ok   Thm 15.6(3) Fisher as minus the expected Hessian  200 trials, tightest margin -1.388e-16
ok   Thm 15.6(2) Fisher under a change of chart      200 trials, tightest margin -5.969e-16


## Chapters 22 to 25: forms, Stokes, divergence, flows

These are the tests that put the orientation and sign conventions of
Part V to work. Stokes' theorem is integrated on the unit disc and the
divergence theorem on the unit ball, by quadrature rather than by sampling,
so the two sides agree to the accuracy of the quadrature: any sign slip in
the boundary orientation, in the exterior derivative or in the definition of
the flux would show up as a factor of $-1$, not as a small discrepancy.

The angle form is checked to be closed everywhere it is defined and to have
circulation $2\pi$ on a loop around the origin, which is why it is not
exact. Chapter 25's adjoint equation is integrated backwards and its
$\alpha(0)$ compared with a finite-difference gradient of the loss, together
with the parameter gradient of part (2).

In [16]:
def t_angle_form(rng):
    """Example 22.15: the angle form on the punctured plane is closed, and
    its integral on a loop around the origin is 2 pi, so it is not exact."""
    # omega = (-y dx + x dy)/(x^2 + y^2)
    f = lambda x, y: -y / (x * x + y * y)
    g = lambda x, y: x / (x * x + y * y)
    # closed: d_x g - d_y f = 0 away from the origin
    for _ in range(20):
        v = rng.normal(size=2)
        v = v / np.linalg.norm(v) * rng.uniform(0.5, 3.0)
        x, y = v
        t = 1e-6
        dxg = (g(x + t, y) - g(x - t, y)) / (2 * t)
        dyf = (f(x, y + t) - f(x, y - t)) / (2 * t)
        scale = max(1.0, abs(dxg), abs(dyf))
        if abs(dxg - dyf) > 1e-7 * scale:
            return False, -abs(dxg - dyf) / scale
    # a loop around the origin: an ellipse, shifted but still enclosing 0
    a, b = rng.uniform(0.5, 2), rng.uniform(0.5, 2)
    N = 20000
    th = np.arange(N) * 2 * np.pi / N
    x, y = a * np.cos(th), b * np.sin(th)
    dx, dy = -a * np.sin(th), b * np.cos(th)
    val = float(np.sum(f(x, y) * dx + g(x, y) * dy) * 2 * np.pi / N)
    return abs(val - 2 * np.pi) < 1e-8, -abs(val - 2 * np.pi)


def t_closed_is_gradient(rng):
    """Ch. 22 games thread: for a field on the plane, d(xi) = 0 exactly when
    xi is the differential of a function."""
    A = rng.normal(size=(2, 2))
    # xi(x) = A x, as a covector field; d xi = 0 iff A is symmetric
    sym = np.max(np.abs(A - A.T)) < 1e-12
    # a potential exists iff the line integral on every loop vanishes
    N = 4000
    th = np.arange(N) * 2 * np.pi / N
    x, y = np.cos(th), 2 * np.sin(th)
    dx, dy = -np.sin(th), 2 * np.cos(th)
    xi1 = A[0, 0] * x + A[0, 1] * y
    xi2 = A[1, 0] * x + A[1, 1] * y
    circ = float(np.sum(xi1 * dx + xi2 * dy) * 2 * np.pi / N)
    exact = abs(circ) < 1e-8
    Asym = (A + A.T) / 2
    if not sym:
        # the antisymmetric part is what the circulation measures
        c = (A[1, 0] - A[0, 1])
        area = np.pi * 1 * 2
        if abs(circ - c * area) > 1e-6 * max(1.0, abs(c * area)):
            return False, -abs(circ - c * area)
    return exact == sym, 0.0 if exact == sym else -1.0


run("Ex 22.15   the angle form is closed, not exact", t_angle_form, 30)
run("Ch. 22     closed means gradient", t_closed_is_gradient, 60)

ok   Ex 22.15   the angle form is closed, not exact   30 trials, tightest margin -0.000e+00
ok   Ch. 22     closed means gradient                 60 trials, tightest margin 0.000e+00


In [17]:
def t_stokes_2d(rng):
    """Thm 23.16 for n = 2 on the unit disc: the integral of d omega over the
    disc equals the circulation of omega on the circle, counterclockwise."""
    c = rng.normal(size=6) * 0.7
    f = lambda x, y: c[0] * x + c[1] * y * y + c[2] * np.sin(x * y)
    g = lambda x, y: c[3] * y + c[4] * x * x + c[5] * np.cos(x - y)
    nr, nt = 300, 2000
    xr, wr = np.polynomial.legendre.leggauss(nr)
    r = (xr + 1) / 2
    wr = wr / 2
    th = np.arange(nt) * 2 * np.pi / nt
    R, T = np.meshgrid(r, th, indexing="ij")
    X, Y = R * np.cos(T), R * np.sin(T)
    h = 1e-5
    dxg = (g(X + h, Y) - g(X - h, Y)) / (2 * h)
    dyf = (f(X, Y + h) - f(X, Y - h)) / (2 * h)
    area = float(np.sum((wr[:, None] * R) * (dxg - dyf)) * 2 * np.pi / nt)
    x, y = np.cos(th), np.sin(th)
    dx, dy = -np.sin(th), np.cos(th)
    circ = float(np.sum(f(x, y) * dx + g(x, y) * dy) * 2 * np.pi / nt)
    err = abs(area - circ) / max(1.0, abs(circ))
    return err < 1e-6, -err


def t_divergence_3d(rng):
    """Thm 24.12 on the unit ball: the integral of div X equals the flux
    of X through the sphere, with the outward normal."""
    c = rng.normal(size=6) * 0.6
    X = lambda P: np.stack([
        c[0] * P[0] + c[1] * P[1] * P[2],
        c[2] * P[1] + c[3] * np.sin(P[0]),
        c[4] * P[2] + c[5] * P[0] * P[1]])
    nr, nth, nph = 80, 160, 160
    xr, wr = np.polynomial.legendre.leggauss(nr)
    r = (xr + 1) / 2
    wr = wr / 2
    xp, wp = np.polynomial.legendre.leggauss(nph)
    phi = (xp + 1) / 2 * np.pi
    wphi = wp / 2 * np.pi
    th = np.arange(nth) * 2 * np.pi / nth
    wth = 2 * np.pi / nth
    R, PH, TH = np.meshgrid(r, phi, th, indexing="ij")
    P = np.stack([R * np.sin(PH) * np.cos(TH),
                  R * np.sin(PH) * np.sin(TH),
                  R * np.cos(PH)])
    h = 1e-5
    div = np.zeros_like(R)
    for i in range(3):
        e = np.zeros((3, 1, 1, 1))
        e[i] = h
        div += (X(P + e)[i] - X(P - e)[i]) / (2 * h)
    vol = float(np.sum((wr[:, None, None] * R ** 2)
                       * (wphi[None, :, None] * np.sin(PH))
                       * div) * wth)
    PH2, TH2 = np.meshgrid(phi, th, indexing="ij")
    S = np.stack([np.sin(PH2) * np.cos(TH2),
                  np.sin(PH2) * np.sin(TH2),
                  np.cos(PH2)])
    flux = float(np.sum((wphi[:, None] * np.sin(PH2))
                        * np.sum(X(S) * S, axis=0)) * wth)
    err = abs(vol - flux) / max(1.0, abs(flux))
    return err < 1e-5, -err


run("Thm 23.16  Stokes on the disc", t_stokes_2d, 8)
run("Thm 24.12  the divergence theorem on the ball", t_divergence_3d, 4)

ok   Thm 23.16  Stokes on the disc                     8 trials, tightest margin -2.799e-14


ok   Thm 24.12  the divergence theorem on the ball     4 trials, tightest margin -9.870e-13


In [18]:
def t_stein(rng):
    """Ch. 23 thread: E[div X + <grad ln p, X>] = 0 for a Gaussian p,
    which is div_p X of Ch. 24."""
    n = int(rng.integers(1, 4))
    S = spd(rng, n)
    m = rng.normal(size=n) * 0.5
    Si = np.linalg.inv(S)
    A = rng.normal(size=(n, n))
    b = rng.normal(size=n)
    Xf = lambda x: A @ x + (b[:, None] if np.ndim(x) > 1 else b)
    # div X = tr A ; <grad ln p, X> = -(x - m)^T S^-1 (A x + b)
    # E[-(x-m)^T S^-1 (A x + b)] = -tr(S^-1 A S)/1 ... computed exactly:
    # with z = x - m, E[z^T S^-1 (A z + A m + b)] = tr(S^-1 A S) = tr(A)
    expect = np.trace(A) - np.trace(Si @ A @ S)
    if abs(expect) > 1e-10 * max(1.0, abs(np.trace(A))):
        return False, -abs(expect)
    # and by Monte Carlo on the whole expression
    N = 400000
    L = np.linalg.cholesky(S)
    z = rng.normal(size=(N, n)) @ L.T + m
    vals = np.trace(A) - np.einsum("ij,jk,ik->i", z - m, Si, Xf(z.T).T)
    mc = vals.mean()
    se = vals.std() / np.sqrt(N)
    return abs(mc) < 5 * se + 1e-9, 5 * se - abs(mc)


def t_div_weighted(rng):
    """Section 24.5: div_p X = div X + <grad ln p, X>, and the weighted
    integration by parts that follows."""
    n = int(rng.integers(1, 4))
    S = spd(rng, n)
    Si = np.linalg.inv(S)
    A = rng.normal(size=(n, n)) * 0.4
    b = rng.normal(size=n) * 0.4
    c = rng.normal(size=n) * 0.4
    Xf = lambda x: A @ x + b
    logp = lambda x: -0.5 * x @ Si @ x
    f = lambda x: np.cos(c @ x)
    gradf = lambda x: -np.sin(c @ x) * c
    x = rng.normal(size=n) * 0.5
    divX = np.trace(A)
    claim = divX + grad_num(logp, x) @ Xf(x)
    # div_p X is the divergence with respect to the volume p vol:
    # div_p X = (1/p) div(p X)
    p = lambda y: np.exp(logp(y))
    h = 1e-5
    direct = 0.0
    for i in range(n):
        e = np.zeros(n)
        e[i] = h
        direct += (p(x + e) * Xf(x + e)[i] - p(x - e) * Xf(x - e)[i]) / (2 * h)
    direct /= p(x)
    e1 = abs(direct - claim) / max(1.0, abs(claim))
    # the weighted identity, by Monte Carlo
    N = 200000
    L = np.linalg.cholesky(S)
    z = rng.normal(size=(N, n)) @ L.T
    lhs = np.array([gradf(zz) @ Xf(zz) for zz in z[:4000]])
    rhs = np.array([-f(zz) * (np.trace(A) - zz @ Si @ Xf(zz)) for zz in z[:4000]])
    d = lhs - rhs
    se = d.std() / np.sqrt(len(d))
    return e1 < 1e-5 and abs(d.mean()) < 5 * se + 1e-6, -e1


run("Ch. 23     Stein's identity", t_stein, 6)
run("Sec 24.5   the weighted divergence", t_div_weighted, 6)

ok   Ch. 23     Stein's identity                       6 trials, tightest margin 4.493e-03


ok   Sec 24.5   the weighted divergence                6 trials, tightest margin -3.194e-11


In [19]:
def t_flow_group(rng):
    """Thm 25.10: phi_{s+t} = phi_s . phi_t, the e^{L|t|} Lipschitz bound,
    and Phi'(t) = DX(phi_t(a)) Phi(t)."""
    n = int(rng.integers(1, 4))
    A = rng.normal(size=(n, n)) * 0.5
    c = rng.normal(size=n) * 0.2
    Xf = lambda y: A @ y + c * np.tanh(np.sum(y))
    DX = lambda y: A + np.outer(c, np.ones(n)) * (1 - np.tanh(np.sum(y)) ** 2)
    F = lambda t, y: Xf(y)
    a = rng.normal(size=n) * 0.3
    s, t = 0.3, 0.5
    p1 = rk4(F, a, 0, s + t, 3000)
    p2 = rk4(F, rk4(F, a, 0, t, 2000), 0, s, 2000)
    e1 = np.max(np.abs(p1 - p2)) / max(1.0, np.max(np.abs(p1)))
    # the Lipschitz bound with L a bound for ||DX|| on the segment
    bb = rng.normal(size=n) * 0.3
    L = max(np.linalg.norm(DX(a + u * (bb - a)), 2) for u in np.linspace(0, 1, 21))
    L = max(L, np.linalg.norm(DX(p1), 2))
    qa = rk4(F, a, 0, t, 2000)
    qb = rk4(F, bb, 0, t, 2000)
    m2 = np.exp(L * t) * np.linalg.norm(a - bb) - np.linalg.norm(qa - qb)
    # the variational equation
    def G(tt, z):
        y = z[:n]
        Phi = z[n:].reshape(n, n)
        return np.concatenate([Xf(y), (DX(y) @ Phi).ravel()])
    z = rk4(G, np.concatenate([a, np.eye(n).ravel()]), 0, t, 4000)
    Phi = z[n:].reshape(n, n)
    hh = 1e-5
    num = np.zeros((n, n))
    for j in range(n):
        e = np.zeros(n)
        e[j] = hh
        num[:, j] = (rk4(F, a + e, 0, t, 2000) - rk4(F, a - e, 0, t, 2000)) / (2 * hh)
    e3 = np.max(np.abs(num - Phi)) / max(1.0, np.max(np.abs(Phi)))
    return e1 < 1e-7 and m2 >= -1e-9 and e3 < 1e-5, min(-e1, m2, -e3)


def t_adjoint_equation(rng):
    """Prop 25.19: alpha' = -alpha . DX(x(t)) with alpha(T) = dl(x(T))
    gives alpha(0) = d(l . phi_T)(a); and the parameter gradient is the
    integral of alpha . D_theta X."""
    n = int(rng.integers(1, 4))
    A = rng.normal(size=(n, n)) * 0.5
    th = rng.normal(size=n) * 0.4
    Xf = lambda y, p: A @ y + p * np.tanh(np.sum(y))
    DX = lambda y, p: A + np.outer(p, np.ones(n)) * (1 - np.tanh(np.sum(y)) ** 2)
    DthX = lambda y, p: np.diag(np.full(n, np.tanh(np.sum(y))))
    w = rng.normal(size=n)
    ell = lambda y: w @ y + 0.3 * (y @ y)
    dell = lambda y: w + 0.6 * y
    T = 0.6
    a = rng.normal(size=n) * 0.3

    def fwd(aa, pp, steps=4000):
        return rk4(lambda t, y: Xf(y, pp), aa, 0, T, steps)

    xT = fwd(a, th)
    # backward: state and adjoint together, and the integral for theta
    def back(t, z):
        y = z[:n]
        al = z[n:2 * n]
        return np.concatenate([-Xf(y, th),
                               DX(y, th).T @ al,
                               DthX(y, th).T @ al])
    z0 = np.concatenate([xT, dell(xT), np.zeros(n)])
    z = rk4(back, z0, 0, T, 6000)
    alpha0, dtheta = z[n:2 * n], z[2 * n:]
    g_num = grad_num(lambda aa: ell(fwd(aa, th)), a, 1e-5)
    e1 = np.max(np.abs(alpha0 - g_num)) / max(1.0, np.max(np.abs(g_num)))
    gt_num = grad_num(lambda pp: ell(fwd(a, pp)), th, 1e-5)
    e2 = np.max(np.abs(dtheta - gt_num)) / max(1.0, np.max(np.abs(gt_num)))
    return max(e1, e2) < 1e-4, -max(e1, e2)


run("Thm 25.10  the flow is a group of diffeomorphisms", t_flow_group, 10)
run("Prop 25.19 the adjoint equation", t_adjoint_equation, 8)

ok   Thm 25.10  the flow is a group of diffeomorphisms   10 trials, tightest margin -8.199e-11


ok   Prop 25.19 the adjoint equation                   8 trials, tightest margin -1.936e-10


## Chapters 16 to 20: measure and integration

Here the tests are of two kinds. Some are inequalities to be violated if
they can be — Markov, Jensen, H\"older, Minkowski, the inclusion of the
$L^p$ on a probability space, Chebyshev behind the Monte Carlo bound —
and random instances look for the violation. The others are the places
where a hypothesis is doing work, and the test is run on both sides of
it: the monotone convergence theorem on $\min(n, x^{-1/2})$, which
climbs to an integral of exactly $2$; Fatou on $n\,1_{(0, 1/n)}$, where
every integral is $1$ and the limit is $0$, so the inequality is strict;
dominated convergence both with a dominating function and, on
$nxe^{-nx^2}$, without one, where the integrals stay at $\frac12$ while
the functions go to zero; and Fubini on the double series that sums to
$1$ one way round and to $0$ the other.

In [20]:
def t_continuity(rng):
    """Thm 16.9: continuity from below always, from above when the first
    set has finite measure -- and not otherwise."""
    # from below: A_n = (0, 1 - 1/n] under Lebesgue measure
    vals = [1 - 1 / n for n in range(1, 400)]
    if abs(vals[-1] - 1.0) > 1e-2:
        return False, -1.0
    # from above with finite measure: A_n = (0, 1/n]
    dec = [1 / n for n in range(1, 2000)]
    if abs(dec[-1]) > 1e-3:
        return False, -1.0
    # from above without finite measure: A_n = [n, inf) has measure inf,
    # but the intersection is empty
    inf_meas = [np.inf for _ in range(5)]
    empty = 0.0
    ok = all(np.isinf(v) for v in inf_meas) and empty == 0.0
    # a finite instance of the same failure: counting measure on {n, n+1, ...}
    return ok, 1.0


def t_linear_image(rng):
    """Thm 16.21: lambda(T(A)) = |det T| lambda(A)."""
    n = int(rng.integers(1, 4))
    T = rng.normal(size=(n, n))
    if abs(np.linalg.det(T)) < 0.05:
        return True, 1.0
    N = 400000
    # A = the unit cube; estimate lambda(T(A)) by rejection in the exact
    # bounding box of the image, found from the vertices of the cube
    verts = np.array([[(k >> j) & 1 for j in range(n)] for k in range(2 ** n)],
                     float)
    V = verts @ T.T
    lo, hi = V.min(axis=0) - 1e-9, V.max(axis=0) + 1e-9
    box = float(np.prod(hi - lo))
    Z = rng.uniform(lo, hi, size=(N, n))
    W = Z @ np.linalg.inv(T).T                      # pull back to the cube
    inside = np.all((W >= 0) & (W <= 1), axis=1)
    est = box * inside.mean()
    se = box * np.sqrt(inside.mean() * (1 - inside.mean()) / N)
    truth = abs(np.linalg.det(T))
    # the sampling error, plus a floor: when the image nearly fills its
    # bounding box the acceptance rate is almost 1 and se collapses
    tol = 5 * se + 1e-3 * truth
    return abs(est - truth) < tol, tol - abs(est - truth)


run("Thm 16.9  continuity of measures", t_continuity, 1)
run("Thm 16.21 the image of a set by a linear map", t_linear_image, 8)

ok   Thm 16.9  continuity of measures                  1 trials, tightest margin 1.000e+00


ok   Thm 16.21 the image of a set by a linear map      8 trials, tightest margin 2.393e-04


In [21]:
def t_dyadic(rng):
    """Lem 17.4: s_n = min(n, 2^-n floor(2^n f)) is simple, increases,
    converges to f, and f - s_n <= 2^-n where f <= n."""
    f = np.concatenate([rng.exponential(2.0, 400), rng.uniform(0, 30, 200)])
    prev = None
    worst = np.inf
    for n in range(1, 16):
        s = np.minimum(n, 2.0 ** -n * np.floor(2.0 ** n * f))
        if prev is not None and np.any(s < prev - 1e-12):
            return False, -1.0
        small = f <= n
        if np.any(f[small] - s[small] > 2.0 ** -n + 1e-12):
            return False, -1.0
        worst = min(worst, float(np.min(2.0 ** -n - (f[small] - s[small]))))
        prev = s
    # convergence where f is finite
    s = np.minimum(40, 2.0 ** -40 * np.floor(2.0 ** 40 * f))
    return np.max(np.abs(f - s)) < 1e-9, worst


def t_convergence_theorems(rng):
    """Thm 17.5 (monotone), 17.13 (Fatou, with strict inequality) and
    17.14 (dominated, and what goes wrong without a dominating function)."""
    # monotone: f_n = min(n, 1/sqrt(x)) increases to 1/sqrt(x) on (0, 1),
    # whose integral is 2.  The substitution x = u^2 makes the quadrature
    # exact: int_0^1 x^{-1/2} dx = int_0^1 2 du.
    u, wu = gauss(400, 0.0, 1.0)
    prev, vals = 0.0, []
    for n in (2, 10, 100, 10000, 10 ** 8):
        val = float(np.sum(wu * np.minimum(n, 1 / u) * 2 * u))
        if val > 2.0 + 1e-6 or val < prev - 1e-9:
            return False, -1.0
        prev = val
        vals.append(val)
    if abs(vals[-1] - 2.0) > 1e-3:
        return False, -abs(vals[-1] - 2.0)

    # Fatou, with strict inequality: f_n = n on (0, 1/n) and 0 elsewhere.
    # Every integral is 1, the pointwise limit is 0, and 0 < 1.
    ints = [n * (1 / n) for n in (1, 2, 10, 1000)]
    if not all(abs(v - 1.0) < 1e-12 for v in ints):
        return False, -1.0
    if not 0.0 < min(ints):
        return False, -1.0

    # dominated, with a dominating function: |e^{-x^2} sin(nx)/n| <= e^{-x^2}
    x, w = gauss(2000, 0.0, 8.0)
    dom = [float(abs(np.sum(w * np.exp(-x ** 2) * np.sin(n * x) / n)))
           for n in (10, 100, 1000)]
    if not (dom[0] > dom[-1] and dom[-1] < 1e-3):
        return False, -dom[-1]

    # without one: f_n = n x e^{-n x^2} tends to 0 at every point of (0, 1),
    # but every integral is about 1/2
    und = []
    for n in (10, 100, 1000, 10000):
        fn = np.minimum(n, n * u ** 2) * 0 + n * u * np.exp(-n * u ** 2)
        und.append(float(np.sum(wu * fn)))
    exact = [0.5 * (1 - np.exp(-n)) for n in (10, 100, 1000, 10000)]
    if max(abs(a - b) for a, b in zip(und, exact)) > 1e-6:
        return False, -1.0
    return abs(und[-1] - 0.5) < 1e-3, abs(und[-1] - 0.5)


def t_markov(rng):
    """Prop 17.8: mu{f >= t} <= (1/t) int f."""
    N = 200000
    z = rng.exponential(1.0, N)
    worst = np.inf
    for t in (0.5, 1.0, 2.0, 5.0):
        lhs = float(np.mean(z >= t))
        rhs = float(np.mean(z)) / t
        worst = min(worst, rhs - lhs)
    return worst >= 0, worst


def t_differentiation(rng):
    """Thm 17.19: differentiation under the integral sign."""
    a = float(rng.uniform(0.5, 2.0))
    x, w = gauss(600, 0.0, 6.0)
    F = lambda s: float(np.sum(w * np.exp(-s * x ** 2) * np.cos(x)))
    h = 1e-5
    num = (F(a + h) - F(a - h)) / (2 * h)
    claim = float(np.sum(w * (-x ** 2) * np.exp(-a * x ** 2) * np.cos(x)))
    err = abs(num - claim) / max(1.0, abs(claim))
    return err < 1e-6, -err


run("Lem 17.4  cutting the range", t_dyadic, 40)
run("Thm 17.5/13/14 monotone, Fatou, dominated", t_convergence_theorems, 2)
run("Prop 17.8  Markov's inequality", t_markov, 10)
run("Thm 17.19 differentiation under the integral", t_differentiation, 40)

ok   Lem 17.4  cutting the range                      40 trials, tightest margin 2.103e-10


ok   Thm 17.5/13/14 monotone, Fatou, dominated         2 trials, tightest margin 1.232e-14
ok   Prop 17.8  Markov's inequality                   10 trials, tightest margin 1.927e-01


ok   Thm 17.19 differentiation under the integral     40 trials, tightest margin -1.545e-10


In [22]:
def t_layer_cake(rng):
    """Prop 18.7: int f = int_0^infty mu{f > t} dt."""
    c = rng.uniform(0.5, 2.0, 3)
    x, w = gauss(800, 0.0, 1.0)
    f = c[0] * x ** 2 + c[1] * np.sin(3 * x) ** 2 + c[2] * x
    direct = float(np.sum(w * f))
    tmax = float(f.max()) * 1.000001
    t, wt = gauss(2000, 0.0, tmax)
    layer = float(np.sum(wt * np.array([np.sum(w * (f > tt)) for tt in t])))
    err = abs(direct - layer) / max(1.0, abs(direct))
    return err < 2e-3, -err


def t_fubini(rng):
    """Thm 18.8 and 18.9: Tonelli and Fubini, and the classic failure without
    integrability."""
    c = rng.normal(size=4)
    f = lambda X, Y: np.exp(-(X ** 2 + Y ** 2)) * (c[0] + c[1] * X + c[2] * Y
                                                   + c[3] * X * Y)
    x, w = gauss(300, -6.0, 6.0)
    X, Y = np.meshgrid(x, x, indexing="ij")
    full = float(np.sum((w[:, None] * w[None, :]) * f(X, Y)))
    inner_first = float(np.sum(w * np.array([np.sum(w * f(np.full_like(x, xi), x))
                                             for xi in x])))
    e1 = abs(full - inner_first) / max(1.0, abs(full))
    # the classic failure: g(x, y) = (x^2 - y^2)/(x^2 + y^2)^2 on (0,1)^2
    # Fubini needs integrability: for the double series a(m, n) that is
    # 1 on the diagonal and -1 just below it, the two iterated sums differ
    K = 400
    a = np.zeros((K, K))
    np.fill_diagonal(a, 1.0)
    a[1:, :-1][np.arange(K - 1), np.arange(K - 1)] = -1.0
    rows_first = float(np.sum(a.sum(axis=1)[:-1]))   # sum over n, then m
    cols_first = float(np.sum(a.sum(axis=0)[:-1]))   # sum over m, then n
    return (e1 < 1e-8 and abs(rows_first - 1.0) < 1e-12
            and abs(cols_first) < 1e-12), -e1


def t_monte_carlo(rng):
    """Thm 18.14: the estimator is unbiased, its variance is sigma^2/N, and
    Chebyshev's bound holds."""
    N = 400
    reps = 4000
    z = rng.exponential(1.5, size=(reps, N))
    g = lambda u: np.sqrt(u)
    m = 1.5 ** 0.5 * np.sqrt(np.pi) / 2          # E[sqrt(Exp(mean 1.5))]
    est = g(z).mean(axis=1)
    sig2 = g(rng.exponential(1.5, 4000000)).var()
    if abs(est.mean() - m) > 4 * np.sqrt(sig2 / (N * reps)):
        return False, -1.0
    ratio = est.var() / (sig2 / N)
    if not 0.9 < ratio < 1.1:
        return False, -abs(ratio - 1)
    eps = 0.05
    emp = float(np.mean(np.abs(est - m) >= eps))
    return emp <= sig2 / (N * eps ** 2) + 1e-12, sig2 / (N * eps ** 2) - emp


def t_convolution(rng):
    """Thm 18.17: the density of a sum is the convolution of the densities."""
    a, b = rng.uniform(0.5, 2.0), rng.uniform(0.5, 2.0)
    # two independent Gaussians: the sum is Gaussian with variance a^2+b^2
    z = float(rng.normal())
    x, w = gauss(900, -14.0, 14.0)
    pa = np.exp(-x ** 2 / (2 * a ** 2)) / (a * np.sqrt(2 * np.pi))
    pb = np.exp(-(z - x) ** 2 / (2 * b ** 2)) / (b * np.sqrt(2 * np.pi))
    conv = float(np.sum(w * pa * pb))
    s2 = a ** 2 + b ** 2
    truth = np.exp(-z ** 2 / (2 * s2)) / np.sqrt(2 * np.pi * s2)
    err = abs(conv - truth) / max(1e-12, truth)
    return err < 1e-8, -err


run("Prop 18.7  the area under the graph", t_layer_cake, 6)
run("Thm 18.8/9 Tonelli and Fubini", t_fubini, 6)
run("Thm 18.14 Monte Carlo", t_monte_carlo, 2)
run("Thm 18.17 convolution of densities", t_convolution, 60)

ok   Prop 18.7  the area under the graph               6 trials, tightest margin -2.234e-05
ok   Thm 18.8/9 Tonelli and Fubini                     6 trials, tightest margin -1.707e-16


ok   Thm 18.14 Monte Carlo                             2 trials, tightest margin 2.397e-01


ok   Thm 18.17 convolution of densities               60 trials, tightest margin -2.568e-14


In [23]:
def t_jensen(rng):
    """Thm 19.1: phi(E[Z]) <= E[phi(Z)], with equality only for a
    constant Z when phi is strictly convex."""
    N = 200000
    z = rng.gamma(2.0, 1.5, N)
    worst = np.inf
    for phi in (np.exp, lambda u: u ** 2, lambda u: -np.log(u),
                lambda u: u * np.log(u)):
        worst = min(worst, float(np.mean(phi(z)) - phi(np.mean(z))))
    const = np.full(N, 2.3)
    eq = abs(np.mean(np.exp(const)) - np.exp(np.mean(const)))
    return worst > 0 and eq < 1e-9, worst


def t_holder_minkowski(rng):
    """Thm 19.3, 19.4 and 19.5: Holder and Minkowski, with equality in Holder when
    |f|^p and |g|^q are proportional."""
    n = 400
    p = float(rng.uniform(1.1, 6.0))
    q = p / (p - 1)
    f = rng.exponential(1.0, n)
    g = rng.exponential(1.0, n)
    nf = lambda u, r: (np.mean(np.abs(u) ** r)) ** (1 / r)
    h1 = nf(f, p) * nf(g, q) - np.mean(np.abs(f * g))
    m1 = nf(f, p) + nf(g, p) - nf(f + g, p)
    # equality case: g = f^{p-1}
    g2 = f ** (p - 1)
    h2 = abs(nf(f, p) * nf(g2, q) - np.mean(f * g2))
    # inclusion on a probability space: ||f||_p <= ||f||_r for p <= r
    r = p + float(rng.uniform(0.1, 3.0))
    inc = nf(f, r) - nf(f, p)
    return (h1 >= -1e-9 and m1 >= -1e-9 and h2 < 1e-8 * max(1.0, nf(f, p))
            and inc >= -1e-9), min(h1, m1, inc)


def t_condexp(rng):
    """Thm 19.9: the conditional expectation given a partition is the
    step function of the block averages, and it is the L^2 projection."""
    n = 2000
    k = int(rng.integers(2, 8))
    blocks = rng.integers(0, k, n)
    z = rng.normal(size=n) * 2 + blocks
    ce = np.array([z[blocks == j].mean() for j in range(k)])[blocks]
    base = float(np.mean((z - ce) ** 2))
    best = base
    for _ in range(60):
        other = rng.normal(size=k)[blocks]
        for s in (0.1, 0.5, 1.0):
            best = min(best, float(np.mean((z - (ce + s * (other - ce))) ** 2)))
    # orthogonality: the residual is uncorrelated with every block function
    orth = max(abs(float(np.mean((z - ce) * (blocks == j)))) for j in range(k))
    return best >= base - 1e-9 and orth < 1e-12, best - base


def t_kl(rng):
    """Thm 19.16: D(Q||P) >= 0, zero only if Q = P, and invariant under a
    measurable bijection."""
    k = int(rng.integers(2, 8))
    q = rng.dirichlet(np.ones(k))
    p = rng.dirichlet(np.ones(k))
    kl = float(np.sum(q * np.log(q / p)))
    if kl < -1e-12:
        return False, kl
    if abs(float(np.sum(q * np.log(q / q)))) > 1e-12:
        return False, -1.0
    perm = rng.permutation(k)                       # a measurable bijection
    kl2 = float(np.sum(q[perm] * np.log(q[perm] / p[perm])))
    return abs(kl - kl2) < 1e-12 and kl > 0, kl


run("Thm 19.1  Jensen", t_jensen, 6)
run("Thm 19.3/4/5 Holder, Minkowski, inclusion", t_holder_minkowski)
run("Thm 19.9  conditional expectation in L^2", t_condexp, 20)
run("Thm 19.16 the Kullback-Leibler divergence", t_kl)

ok   Thm 19.1  Jensen                                  6 trials, tightest margin 2.689e-01
ok   Thm 19.3/4/5 Holder, Minkowski, inclusion       200 trials, tightest margin 3.607e-02
ok   Thm 19.9  conditional expectation in L^2         20 trials, tightest margin 0.000e+00
ok   Thm 19.16 the Kullback-Leibler divergence       200 trials, tightest margin 1.430e-02


## Chapter 20 and Appendix A

The volume of the ball is checked twice, once through the polar formula
and the Gamma function and once by counting points, so that an error in
either would show. The two lemmas of the appendix are the ones the
construction of Lebesgue measure turns on: that a half-space cuts a box
into two boxes whose volumes add, and that counting the points of
$\frac1N\Z^n$ in a box and dividing by $N^n$ converges to its volume.

In [24]:
def t_polar(rng):
    """Thm 20.13: the polar formula, and the volume of the ball."""
    n = int(rng.integers(2, 7))
    from math import gamma, pi
    area = 2 * pi ** (n / 2) / gamma(n / 2)
    R = float(rng.uniform(0.5, 2.0))
    vol = pi ** (n / 2) / gamma(n / 2 + 1) * R ** n
    if abs(vol - area / n * R ** n) > 1e-9 * vol:
        return False, -1.0
    # the Gaussian integral through the polar formula
    r, w = gauss(900, 0.0, 30.0)
    g = float(np.sum(w * np.exp(-r ** 2 / 2) * r ** (n - 1))) * area
    truth = (2 * pi) ** (n / 2)
    e1 = abs(g - truth) / truth
    # the volume by Monte Carlo
    N = 400000
    X = rng.uniform(-R, R, size=(N, n))
    inside = (X ** 2).sum(axis=1) <= R ** 2
    est = (2 * R) ** n * inside.mean()
    se = (2 * R) ** n * np.sqrt(max(inside.mean() * (1 - inside.mean()), 1e-12) / N)
    return e1 < 1e-8 and abs(est - vol) < 5 * se, -e1


def t_pushforward(rng):
    """Thm 20.6: q(g(x)) = p(x)/|det Dg(x)|."""
    n = int(rng.integers(1, 4))
    A = rng.normal(size=(n, n))
    if abs(np.linalg.det(A)) < 0.1:
        return True, 1.0
    b = rng.normal(size=n)
    S = np.eye(n)
    # x ~ N(0, I), y = A x + b has density N(b, A A^T)
    x = rng.normal(size=n)
    y = A @ x + b
    p = np.exp(-x @ x / 2) / (2 * np.pi) ** (n / 2)
    claim = p / abs(np.linalg.det(A))
    C = A @ A.T
    truth = (np.exp(-(y - b) @ np.linalg.solve(C, y - b) / 2)
             / np.sqrt((2 * np.pi) ** n * np.linalg.det(C)))
    err = abs(claim - truth) / max(1e-300, truth)
    return err < 1e-9, -err


run("Thm 20.13 polar coordinates and the ball", t_polar, 6)
run("Thm 20.6  the density of a push-forward", t_pushforward)

ok   Thm 20.13 polar coordinates and the ball          6 trials, tightest margin -5.810e-14
ok   Thm 20.6  the density of a push-forward         200 trials, tightest margin -1.096e-13


In [25]:
def t_halfspace(rng):
    """Lem A.9: a half-space cuts a box into two boxes whose volumes add."""
    n = int(rng.integers(1, 5))
    a = rng.normal(size=n)
    b = a + rng.uniform(0.2, 2.0, n)
    i = int(rng.integers(0, n))
    c = float(rng.uniform(a[i] - 0.5, b[i] + 0.5))
    vol = lambda lo, hi: float(np.prod(np.maximum(hi - lo, 0.0)))
    full = vol(a, b)
    b1 = b.copy()
    b1[i] = min(b[i], c)
    a2 = a.copy()
    a2[i] = max(a[i], c)
    parts = vol(a, b1) + vol(a2, b)
    return abs(parts - full) < 1e-12 * max(1.0, full), -abs(parts - full)


def t_grid_count(rng):
    """Lem A.5: the number of grid points of (1/N)Z^n in a box, divided by
    N^n, tends to the volume."""
    n = int(rng.integers(1, 4))
    a = rng.uniform(-2, 0, n)
    b = a + rng.uniform(0.3, 2.0, n)
    vol = float(np.prod(b - a))
    errs = []
    for N in (50, 200, 800):
        counts = np.prod([np.ceil(N * b[i]) - np.ceil(N * a[i]) for i in range(n)])
        errs.append(abs(counts / N ** n - vol))
    return errs[-1] < 0.02 * max(1.0, vol), -errs[-1]


run("Lem A.9   half-spaces cut boxes into boxes", t_halfspace)
run("Lem A.5   counting grid points", t_grid_count, 40)

ok   Lem A.9   half-spaces cut boxes into boxes      200 trials, tightest margin -4.441e-16
ok   Lem A.5   counting grid points                   40 trials, tightest margin -2.120e-03


## Summary

In [26]:
print(f"{len(summary)} statements, no counterexample\n")
tight = sorted(summary, key=lambda r: abs(r[2]))[:6]
print("the six tightest margins, where the statement is attained:")
for name, trials, worst in tight:
    print(f"  {name:46s} {worst:+.3e}")
assert len(summary) == 63

63 statements, no counterexample

the six tightest margins, where the statement is attained:
  Thm 3.25  the closest solution                 +0.000e+00
  Thm 4.35  projection on a complete subspace    +0.000e+00
  Thm 5.3   Sylvester's law of inertia           +0.000e+00
  Thm 5.20  Eckart-Young-Mirsky                  +0.000e+00
  Prop 5.21 orthogonal Procrustes                +0.000e+00
  Thm 8.6   symmetry of the second differential  -0.000e+00
